# 04 · Apriori: what should be pooled and bundled together?
**Business problem:** every extra category on one delivery run spreads delivery cost and supports the deck's "savings disappear in delivery" mitigation. Each respondent's selected categories act as a basket; a second run uses selected platform features.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False,'font.size':10})
BLUE,ORANGE,AQUA,YELLOW='#2a78d6','#eb6834','#1baf7a','#eda100'
FIG='../Results/figures/'; SEED=42
c=pd.read_csv('../Dataset/customers.csv')
from mlxtend.frequent_patterns import apriori,association_rules
def mine(prefix,sup,conf):
    b=c.filter(like=prefix).astype(bool); b.columns=[x.replace(prefix,'') for x in b.columns]
    fi=apriori(b,min_support=sup,use_colnames=True); r=association_rules(fi,num_itemsets=len(b),metric='confidence',min_threshold=conf)
    r=r[r.lift>1.05].copy(); r['Antecedent']=r.antecedents.map(lambda s:' + '.join(sorted(s))); r['Consequent']=r.consequents.map(lambda s:' + '.join(sorted(s)))
    print(prefix,'baskets',len(b),'frequent itemsets',len(fi),'rules',len(r)); return r[['Antecedent','Consequent','support','confidence','lift','leverage','conviction']].sort_values('lift',ascending=False)
cat=mine('CAT_',.08,.40).assign(Rule_Type='Category bundle'); fe=mine('FEAT_',.10,.45).assign(Rule_Type='Feature pairing')
rules=pd.concat([cat,fe]).round(4); rules.to_csv('../Results/Association_Rules.csv',index=False)
print(cat.head(12).round(3).to_string(index=False)); print(fe.head(8).round(3).to_string(index=False))

CAT_ baskets 1200 frequent itemsets 60 rules 77
FEAT_ baskets 1200 frequent itemsets 90 rules 90
                             Antecedent                              Consequent  support  confidence  lift  leverage  conviction       Rule_Type
 Fresh_Produce_Meat + Groceries_Staples                               Baby_Kids    0.112       0.547 2.723     0.071       1.764 Category bundle
                              Baby_Kids  Fresh_Produce_Meat + Groceries_Staples    0.112       0.556 2.723     0.071       1.792 Category bundle
 Groceries_Staples + Household_Cleaning      Fresh_Produce_Meat + Personal_Care    0.108       0.483 2.697     0.068       1.589 Category bundle
     Fresh_Produce_Meat + Personal_Care  Groceries_Staples + Household_Cleaning    0.108       0.605 2.697     0.068       1.962 Category bundle
          Baby_Kids + Groceries_Staples                      Fresh_Produce_Meat    0.112       0.870 2.643     0.069       5.165 Category bundle
                              Bab

## Business interpretation
- Baby, groceries, fresh produce and household items form a clear family basket, which supports shared-delivery bundles.
- Building pooling, consolidated delivery and WhatsApp updates are chosen together, suggesting one "building campaign" offer.
- **Automation:** when a buyer commits to one category, suggest the highest-lift open campaign. Rules show overlap in stated interest, not purchases or causation.